# Coral Reef Segmentation — SegFormer (MiT-B2) on Coralscapes (Colab)

**Runtime:** `Runtime → Change runtime type → T4 GPU`.

**Data:** Coralscapes (Sauder et al., 2025, EPFL) — https://huggingface.co/datasets/EPFL-ECEO/coralscapes
(Apache-2.0, no login). 2,075 expert-annotated coral reef images (1024×2048) from 35 Red Sea dive sites,
39 classes. Official **spatial** split: train 1,517 (27 sites) · val 166 (3 sites) · test 392 (5 sites) —
test sites are never seen in training. Download ~5.9 GB, straight into Colab.

**Why this replaces SUIM:** SUIM only says *where reef is* (one "reefs & invertebrates" class) and has no
bleaching labels. Coralscapes labels **live, bleached and dead coral** separately, so the model learns bleaching
from expert labels instead of the local colour rule.

**Classes used here** (the 39 classes grouped into 4; id 0 = unlabelled → ignored, as in the paper):
`other` (substrate, sand, rubble, algae, seagrass, animals, fish, humans, transect gear, background) ·
`live_coral` · `bleached_coral` · `dead_coral`. *Coral* = live + bleached + dead.

**Reported on the official TEST split** (never used for training or model selection): 4-class mIoU, coral IoU,
bleached IoU, pixel accuracy, binary coral-vs-rest pixel accuracy, and image-level accuracy for
"is there coral?" / "is there bleaching?". The paper's 39-class mIoU (SegFormer MiT-B2: 0.547) is a
different, harder task and is not comparable to the grouped numbers.

**Quality gate:** coral IoU on TEST ≥ `GATE_CORAL_IOU`. If it fails, the last cell says so and the local project
keeps its current model.

**Output** (download from `MyDrive/coralscapes/` into the local project's `models/coral_segmenter/`):
`coralscapes_segformer.pt`, `coralscapes_segformer_config.json`, `coralscapes_segformer_test_metrics.json`.

In [ ]:
!nvidia-smi
!pip install -q 'segmentation-models-pytorch>=0.5' albumentations datasets

In [ ]:
# ---- Config -------------------------------------------------------------
ARCH      = 'Segformer'
ENCODER   = 'mit_b2'       # same backbone as the paper's SegFormer MiT-B2 baseline
TAG       = 'coralscapes_segformer'
IMG_H, IMG_W = 512, 1024   # Coralscapes frames are 1024x2048; trained at half resolution to fit a T4
CROP      = 512            # random 512x512 crops for training; evaluation uses the full 512x1024 frame
EPOCHS    = 60
PATIENCE  = 12             # early stop on val mIoU
BATCH     = 8
LR        = 6e-5           # encoder LR (paper uses 6e-5 AdamW for SegFormer); decoder head 10x
WARMUP    = 3
SEED      = 42
GATE_CORAL_IOU = 0.50
CORAL_PRESENT_FRAC = 0.02      # image-level: coral if >= 2% of labelled pixels are coral (same 2% as the local app)
BLEACH_PRESENT_FRAC = 0.005    # image-level: bleaching if >= 0.5% of labelled pixels are bleached coral
# -------------------------------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_DIR = '/content/drive/MyDrive/coralscapes'
os.makedirs(DRIVE_DIR, exist_ok=True)

## Download Coralscapes and group the 39 classes
Class ids (1–39) are taken from the dataset card; 0 = unlabelled. The grouping is printed so it can be checked.

In [ ]:
import numpy as np, json, random, torch
from datasets import load_dataset

ds = load_dataset('EPFL-ECEO/coralscapes')        # ~5.9 GB, cached under ~/.cache/huggingface
print(ds)

ID2NAME = {1: 'seagrass', 2: 'trash', 3: 'other coral dead', 4: 'other coral bleached', 5: 'sand',
           6: 'other coral alive', 7: 'human', 8: 'transect tools', 9: 'fish', 10: 'algae covered substrate',
           11: 'other animal', 12: 'unknown hard substrate', 13: 'background', 14: 'dark', 15: 'transect line',
           16: 'massive/meandering bleached', 17: 'massive/meandering alive', 18: 'rubble', 19: 'branching bleached',
           20: 'branching dead', 21: 'millepora', 22: 'branching alive', 23: 'massive/meandering dead', 24: 'clam',
           25: 'acropora alive', 26: 'sea cucumber', 27: 'turbinaria', 28: 'table acropora alive', 29: 'sponge',
           30: 'anemone', 31: 'pocillopora alive', 32: 'table acropora dead', 33: 'meandering bleached',
           34: 'stylophora alive', 35: 'sea urchin', 36: 'meandering alive', 37: 'meandering dead',
           38: 'crown of thorn', 39: 'dead clam'}
CLASSES = ['other', 'live_coral', 'bleached_coral', 'dead_coral']
LIVE, BLEACHED, DEAD = 1, 2, 3
CORAL = [LIVE, BLEACHED, DEAD]
IGNORE = 255

def group(name):
    if 'clam' in name:                       # 'dead clam' is not coral
        return 0
    if 'bleached' in name:
        return BLEACHED
    if 'dead' in name:
        return DEAD
    if 'alive' in name or name in ('millepora', 'turbinaria'):
        return LIVE
    return 0

LUT = np.full(256, IGNORE, np.uint8)
for i, n in ID2NAME.items():
    LUT[i] = group(n)
for g, c in enumerate(CLASSES):
    print(f'{c:15s} <- {[ID2NAME[i] for i in ID2NAME if LUT[i] == g]}')

lbl = np.array(ds['train'][0]['label'])
print('label array', lbl.shape, lbl.dtype, 'ids present:', np.unique(lbl)[:20])
assert lbl.ndim == 2 and lbl.max() <= 39, 'Unexpected label format - check the dataset card'

In [ ]:
import cv2, albumentations as A
from torch.utils.data import Dataset, DataLoader
torch.manual_seed(SEED); random.seed(SEED); np.random.seed(SEED)

MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
train_tf = A.Compose([
    A.Resize(IMG_H, IMG_W), A.RandomScale((0.0, 0.5), p=0.5),
    A.RandomCrop(CROP, CROP), A.HorizontalFlip(p=0.5),
    A.ColorJitter(0.2, 0.2, 0.2, 0.02, p=0.5), A.Normalize(MEAN, STD),
])
eval_tf = A.Compose([A.Resize(IMG_H, IMG_W), A.Normalize(MEAN, STD)])

class Coralscapes(Dataset):
    def __init__(self, split, tf):
        self.d, self.tf = split, tf
    def __len__(self):
        return len(self.d)
    def __getitem__(self, i):
        row = self.d[i]
        img = np.array(row['image'].convert('RGB'))
        mask = LUT[np.array(row['label']).astype(np.uint8)]
        o = self.tf(image=img, mask=mask)
        return torch.from_numpy(o['image'].transpose(2, 0, 1)).float(), torch.from_numpy(o['mask']).long()

dl_train = DataLoader(Coralscapes(ds['train'], train_tf), batch_size=BATCH, shuffle=True, num_workers=2, drop_last=True)
dl_val = DataLoader(Coralscapes(ds['validation'], eval_tf), batch_size=4, num_workers=2)
dl_test = DataLoader(Coralscapes(ds['test'], eval_tf), batch_size=4, num_workers=2)

# Class weights (bleached coral is rare): 1/sqrt(frequency) on a 300-image train sample, normalised to mean 1
counts = np.zeros(len(CLASSES))
for i in random.Random(SEED).sample(range(len(ds['train'])), 300):
    m = LUT[np.array(ds['train'][i]['label']).astype(np.uint8)]
    counts += np.bincount(m[m != IGNORE], minlength=len(CLASSES))
freq = counts / counts.sum()
W = 1 / np.sqrt(np.maximum(freq, 1e-6)); W = W / W.mean()
print('pixel share:', dict(zip(CLASSES, freq.round(4))), '\nclass weights:', dict(zip(CLASSES, W.round(2))))

## Train SegFormer (MiT-B2) — weighted cross-entropy + Dice, best checkpoint by val mIoU

In [ ]:
import segmentation_models_pytorch as smp
device = 'cuda'
model = smp.create_model(ARCH, encoder_name=ENCODER, encoder_weights='imagenet', classes=len(CLASSES)).to(device)
dice = smp.losses.DiceLoss('multiclass', ignore_index=IGNORE)
ce = torch.nn.CrossEntropyLoss(weight=torch.tensor(W, dtype=torch.float32, device=device), ignore_index=IGNORE)

enc_ids = {id(p) for p in model.encoder.parameters()}
opt = torch.optim.AdamW([{'params': list(model.encoder.parameters()), 'lr': LR},
                         {'params': [p for p in model.parameters() if id(p) not in enc_ids], 'lr': LR * 10}], weight_decay=1e-2)
sched = torch.optim.lr_scheduler.SequentialLR(opt, [
    torch.optim.lr_scheduler.LinearLR(opt, start_factor=0.1, total_iters=WARMUP),
    torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS - WARMUP)], milestones=[WARMUP])
scaler = torch.amp.GradScaler('cuda')
K = len(CLASSES)

def evaluate(loader):
    # Pixel confusion matrix (labelled pixels only) + per-image coral / bleaching fractions (truth, prediction)
    model.eval()
    cm = torch.zeros(K, K, dtype=torch.long, device=device)
    per_image = []
    with torch.no_grad(), torch.autocast('cuda'):
        for x, y in loader:
            p = model(x.to(device)).argmax(1); y = y.to(device)
            valid = y != IGNORE
            cm += torch.bincount(y[valid] * K + p[valid], minlength=K * K).view(K, K)
            for yi, pi, vi in zip(y, p, valid):
                n = vi.sum().clamp(min=1)
                per_image.append([float(((yi == c) & vi).sum() / n) for c in (1, 2, 3)] +
                                 [float(((pi == c) & vi).sum() / n) for c in (1, 2, 3)])
    return cm.cpu().numpy(), np.array(per_image)

def iou_from_cm(cm):
    inter = np.diag(cm); union = cm.sum(0) + cm.sum(1) - inter
    return np.where(union > 0, inter / np.maximum(union, 1), np.nan)

def coral_iou(cm):
    c = np.array(CORAL)
    tp = cm[np.ix_(c, c)].sum(); fp = cm[:, c].sum() - tp; fn = cm[c, :].sum() - tp
    return tp / max(1, tp + fp + fn), tp / max(1, tp + fp), tp / max(1, tp + fn)

CKPT = f'{DRIVE_DIR}/{TAG}.pt'
best_miou, bad = -1, 0
for epoch in range(EPOCHS):
    model.train(); total = 0
    for x, y in dl_train:
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        with torch.autocast('cuda'):
            out = model(x); loss = ce(out, y) + dice(out, y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        total += loss.item()
    sched.step()
    cm, _ = evaluate(dl_val); ious = iou_from_cm(cm); miou = np.nanmean(ious)
    print(f'epoch {epoch+1:3d}  loss {total/len(dl_train):.3f}  val mIoU {miou:.3f}  coral IoU {coral_iou(cm)[0]:.3f}  '
          f'live {ious[LIVE]:.3f}  bleached {ious[BLEACHED]:.3f}  dead {ious[DEAD]:.3f}', flush=True)
    if miou > best_miou:
        best_miou, bad = miou, 0
        torch.save(model.state_dict(), CKPT)
    else:
        bad += 1
        if bad >= PATIENCE:
            print('early stop'); break
print('best val mIoU', round(best_miou, 3))

## Evaluate once on the official TEST split (5 unseen dive sites)

In [ ]:
model.load_state_dict(torch.load(CKPT, map_location=device))
cm, img = evaluate(dl_test)
ious = iou_from_cm(cm)
c_iou, c_prec, c_rec = coral_iou(cm)
c = np.array(CORAL)
binary_acc = (cm[np.ix_(c, c)].sum() + cm[0, 0]) / cm.sum()      # coral vs not-coral, per pixel

def image_level(true_frac, pred_frac, thr):
    t, p = true_frac >= thr, pred_frac >= thr
    tp, fp, fn = int((t & p).sum()), int((~t & p).sum()), int((t & ~p).sum())
    return {'accuracy': round(float((t == p).mean()), 4), 'precision': round(tp / max(1, tp + fp), 4),
            'recall': round(tp / max(1, tp + fn), 4), 'n_positive': int(t.sum()), 'n_images': len(t),
            'always_positive_baseline': round(float(max(t.mean(), 1 - t.mean())), 4)}

true_coral, pred_coral = img[:, 0:3].sum(1), img[:, 3:6].sum(1)
metrics = {
    'dataset': 'Coralscapes TEST (official spatial split, 5 unseen dive sites)', 'model': f'{ARCH} {ENCODER}',
    'classes': CLASSES, 'input_hw': [IMG_H, IMG_W], 'n_test_images': len(ds['test']),
    'pixel_accuracy': round(float(np.diag(cm).sum() / cm.sum()), 4),
    'mean_iou': round(float(np.nanmean(ious)), 4),
    'per_class_iou': {k: (None if np.isnan(v) else round(float(v), 4)) for k, v in zip(CLASSES, ious)},
    'coral_iou': round(float(c_iou), 4), 'coral_precision': round(float(c_prec), 4), 'coral_recall': round(float(c_rec), 4),
    'coral_vs_rest_pixel_accuracy': round(float(binary_acc), 4),
    'image_level_coral_present': image_level(true_coral, pred_coral, CORAL_PRESENT_FRAC),
    'image_level_bleaching_present': image_level(img[:, 1], img[:, 4], BLEACH_PRESENT_FRAC),
    'gate_coral_iou': GATE_CORAL_IOU, 'passed_gate': bool(c_iou >= GATE_CORAL_IOU),
    'note': 'Grouped 4-class task; not comparable to the paper 39-class mIoU (SegFormer MiT-B2 0.547).',
}
# 'reef_*' aliases so the local app/report can treat it like the SUIM models (reef = any coral)
metrics.update(reef_iou=metrics['coral_iou'], reef_precision=metrics['coral_precision'], reef_recall=metrics['coral_recall'])
print(json.dumps(metrics, indent=2))
json.dump(metrics, open(f'{DRIVE_DIR}/{TAG}_test_metrics.json', 'w'), indent=2)
json.dump({'arch': ARCH, 'encoder': ENCODER, 'classes': CLASSES, 'reef_classes': CORAL, 'live_class': LIVE,
           'bleached_class': BLEACHED, 'dead_class': DEAD, 'input_hw': [IMG_H, IMG_W], 'mean': MEAN, 'std': STD,
           'dataset': 'Coralscapes'}, open(f'{DRIVE_DIR}/{TAG}_config.json', 'w'), indent=2)

In [ ]:
# Visual check: 4 TEST images - image, ground truth, prediction (green live, white bleached, brown dead, grey other)
import matplotlib.pyplot as plt
PAL = np.array([[90, 90, 90], [40, 200, 80], [255, 255, 255], [140, 90, 40]], np.uint8)
fig, axes = plt.subplots(4, 3, figsize=(15, 10))
for row, i in enumerate(random.Random(1).sample(range(len(ds['test'])), 4)):
    r = ds['test'][i]
    im = cv2.resize(np.array(r['image'].convert('RGB')), (IMG_W, IMG_H))
    gt = cv2.resize(LUT[np.array(r['label']).astype(np.uint8)], (IMG_W, IMG_H), interpolation=cv2.INTER_NEAREST)
    x = torch.from_numpy(eval_tf(image=im)['image'].transpose(2, 0, 1))[None].float().to(device)
    with torch.no_grad():
        pred = model(x).argmax(1)[0].cpu().numpy()
    gt_rgb = PAL[np.where(gt == IGNORE, 0, gt)]
    for ax, show, t in zip(axes[row], [im, gt_rgb, PAL[pred]], ['image', 'ground truth', 'prediction']):
        ax.imshow(show); ax.set_title(t); ax.axis('off')
plt.tight_layout(); plt.savefig(f'{DRIVE_DIR}/{TAG}_test_examples.png', dpi=100)

In [ ]:
if metrics['passed_gate']:
    print(f"PASS: coral IoU {metrics['coral_iou']:.3f} >= {GATE_CORAL_IOU}. Download {TAG}.pt, {TAG}_config.json and "
          f"{TAG}_test_metrics.json from MyDrive/coralscapes into models/coral_segmenter/.")
else:
    print(f"FAIL: coral IoU {metrics['coral_iou']:.3f} < {GATE_CORAL_IOU}. Options: ENCODER='mit_b3', EPOCHS=100, "
          "IMG_H/W=768/1536 with CROP=768 (needs an A100). The local project keeps its current model; report this result.")